In [ ]:
# =============================================
# PHASE 1 — TRAIN TOP LAYERS
# =============================================
print("\n" + "="*50)
print("PHASE 1 — Training top layers")
print("="*50)

history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=EPOCHS,
    callbacks=callbacks,
    class_weight=class_weight_dict
)


In [ ]:
# =============================================
# PHASE 2 — FINE TUNING
# =============================================
print("\n" + "="*50)
print("PHASE 2 — Fine tuning")
print("="*50)

with strategy.scope():
    base_model.trainable = True
    for layer in base_model.layers[:-30]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"]
    )

history_fine = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=5,
    callbacks=callbacks,
    class_weight=class_weight_dict
)

In [ ]:

# =============================================
# SAVE FINAL MODEL
# =============================================
model.save("/kaggle/working/mars_terrain_multicamera_final.keras")
print("✅ Model saved!")


In [ ]:
# =============================================
# PLOT TRAINING HISTORY
# =============================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

acc      = history.history["accuracy"]     + history_fine.history["accuracy"]
val_acc  = history.history["val_accuracy"] + history_fine.history["val_accuracy"]
loss     = history.history["loss"]         + history_fine.history["loss"]
val_loss = history.history["val_loss"]     + history_fine.history["val_loss"]

axes[0].plot(acc,     label="Train Accuracy")
axes[0].plot(val_acc, label="Val Accuracy")
axes[0].axvline(x=EPOCHS-1, color="red", linestyle="--", label="Fine tune start")
axes[0].set_title("Accuracy")
axes[0].legend()

axes[1].plot(loss,     label="Train Loss")
axes[1].plot(val_loss, label="Val Loss")
axes[1].axvline(x=EPOCHS-1, color="red", linestyle="--", label="Fine tune start")
axes[1].set_title("Loss")
axes[1].legend()

plt.tight_layout()
plt.savefig("/kaggle/working/training_history.png")
plt.show()


In [ ]:
# =============================================
# EVALUATION
# =============================================
val_loss_score, val_acc_score = model.evaluate(val_ds)
print(f"\nFinal Validation Loss    : {val_loss_score:.4f}")
print(f"Final Validation Accuracy: {val_acc_score:.4f}")

In [ ]:
# =============================================
# PER CLASS ACCURACY
# =============================================
y_pred = []
y_true = []

for imgs, labels in val_ds:
    preds = model.predict(imgs, verbose=0)
    y_pred.extend(np.argmax(preds, axis=1))
    y_true.extend(labels.numpy())

print("\nClassification Report:")
print(classification_report(
    y_true, y_pred,
    target_names=list(CLASS_NAMES.values())
))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d",
            xticklabels=CLASS_NAMES.values(),
            yticklabels=CLASS_NAMES.values(),
            cmap="Blues")
plt.title("Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.savefig("/kaggle/working/confusion_matrix.png")
plt.show()

In [ ]:
# Save model as output — persists even after session ends
import shutil
shutil.copy("/kaggle/working/mars_terrain_v2_final.keras", 
            "/kaggle/working/mars_terrain_v2_final_backup.keras")
print("✅ Backup saved!")
